# TRACK 1 — 가설 검증 TOP5 재현 코드
국민여행조사 × 지방재정 상관관계 분석 · 가설1(방문×지출) / 가설2(방문×예산)

이 노트북은 [가설 검증 TOP5 — 이론과 결과] 아티팩트에 사용된 모든 통계치를 원본 파일에서 직접 재현하는 코드입니다.
중간 산출 CSV(파생 파일)를 별도로 만들지 않고, 팀 원본 파일 두 개만 읽어 노트북 안에서 바로 패널을 구성합니다.

**입력 파일 (이 노트북과 같은 폴더, 즉 GitHub 리포 안의 원본 경로)**
- `processed_combined.csv` — 국민여행조사 통합 전처리 최종본(101개 컬럼, 156,050행). `processed_combined.ipynb` 실행 결과물.
- `processed_regional.csv` — 지방재정 세출현황 전처리 최종본(19개 컬럼, 48,617행, subsector 컬럼 포함). `processed_regional.ipynb` 실행 결과물.

**주요 방법론 결정사항 (2026-09-08 업데이트)**
- ~~세부분야(subsector) `문화및관광일반`(065) 전체 제외~~ → **폐기**. 065를 통째로 빼면 서울 -34%, 대구 -31% 등
  지역별 예산 총액이 너무 크게 흔들려서, 065는 **다시 포함**하기로 팀에서 결정.
- 대신 `scope_filter_proposal.html`(관광 연구범위 재검토)에서 사람이 직접 검토해 뽑은 **세부사업명(project_name)
  110건(고유 39개)만 콕 집어 제외** — 노래연습장업 인허가, 시사편찬, 옥외광고물 관리 등 subsector는 관광/문화재/
  문화및관광일반이지만 실질은 관광과 무관한 인접업종 행정 사업. 예산 영향은 지역별 최대 -0.13%(대전) 수준으로 미미.
- 가설1은 세출예산 데이터를 쓰지 않으므로(방문건수·여행지출액만 사용) 이 필터와 무관, 수치 변동 없음
- TOP5 방법(두 가설에 동일 적용): ⑤고정효과 → ⑥변화량분석(Pearson+Spearman) → ⑧순열검정 → ⑦부트스트랩 → ②제주제외


## 0. 라이브러리 임포트

In [4]:
1+1

2

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf

pd.set_option('display.float_format', lambda x: f'{x:.4f}')
RNG_SEED = 42


## 1. 원본 파일 로드 및 시/도×연도(n=51) 패널 구성

`processed_combined.csv`는 응답자 단위(156,050행) 원자료입니다. 여기서 시/도별 사전집계 컬럼
(`trip_cnt_{지역}` = 방문 횟수, `trip_exp_{지역}` = 여행지출액)을 `wt_dom`(개인 가중치)으로 가중합산해
지역×연도 패널을 만듭니다. 이 가중합산 방식은 `processed_combined.ipynb` 6절(공식 통계 대조)에서
전체 관광여행 총량·지출총량을 검증한 것과 동일한 방식(Σ wt_dom × 값)을 지역별로 그대로 적용한 것입니다.


In [3]:
combined = pd.read_csv('processed_combined.csv')
print('processed_combined:', combined.shape)

# processed_combined.ipynb의 영문화 규칙과 동일한 지역명 목록/매핑 (컬럼 정의서 기준)
REGION_EN = ['seoul','busan','daegu','incheon','gwangju','daejeon','ulsan','sejong',
             'gyeonggi','gangwon','chungbuk','chungnam','jeonbuk','jeonnam',
             'gyeongbuk','gyeongnam','jeju']

rows = []
for yr, g in combined.groupby('year'):
    w = g['wt_dom'].values
    for r in REGION_EN:
        visit_w = float((g[f'trip_cnt_{r}'].values * w).sum())
        spend_w = float((g[f'trip_exp_{r}'].values * w).sum())
        rows.append({'year': yr, 'region': r, 'visit_w': visit_w, 'spend_w': spend_w})

panel = pd.DataFrame(rows)
print('panel(시/도x연도):', panel.shape, '(17개 지역 x 3개년 = 51 이어야 함)')
panel.head()


FileNotFoundError: [Errno 2] No such file or directory: 'processed_combined.csv'

## 2. 가설2 예산 변수 재구성 — 관광 연구범위 재검토 반영 (110건 제외, 065는 포함)

`subsector`(관광/문화재/문화및관광일반) 기준 제외는 폐기합니다. 대신 `scope_filter_proposal.html`에서
사람이 `project_name`(세부사업명) 고유값을 전수 검토해 뽑은 **관광과 무관한 인접업종 사업 39개(행 기준 110건)**만
`project_name`으로 직접 제외합니다. subsector는 필터링하지 않으므로 065(문화및관광일반)도 그대로 포함됩니다.


In [ ]:
reg = pd.read_csv('processed_regional.csv')
print('processed_regional:', reg.shape)
print(reg['subsector'].value_counts())

# 한글 시/도명 -> 영문 코드 매핑 (processed_combined.ipynb region_en 딕셔너리와 동일)
REGION_KR2EN = {
    '서울':'seoul','부산':'busan','대구':'daegu','인천':'incheon','광주':'gwangju',
    '대전':'daejeon','울산':'ulsan','세종':'sejong','경기':'gyeonggi','강원':'gangwon',
    '충북':'chungbuk','충남':'chungnam','전북':'jeonbuk','전남':'jeonnam',
    '경북':'gyeongbuk','경남':'gyeongnam','제주':'jeju'
}

# scope_filter_proposal.html(2026-09-08, 팀 검토)에서 확정한 관광 연구범위 제외 대상 39개 세부사업명.
# 노래연습장업 인허가/교육(A), 게임·음악산업 규제·진흥(B), 옥외광고물 관리(C), 지역사 편찬·향토사료(D),
# 기타 인접업종 인허가단속(E), 개별 확인된 오분류 의심 2건(F) — 문서 원문에서 그대로 가져온 리스트.
SCOPE_EXCLUDE_PROJECTS = [
    '노래연습장업·게임제공업 등 관리 운영', '건전한노래연습장질서확립', '노래연습장업자 교육',
    '노래연습장 관련 유해환경 개선', '노래연습장 업소 교육',
    '사행성오락업소지도단속', '건전한 게임산업 및 음악산업 육성지원', '비디오산업게임관리',
    '게임·음반·영상 관련업소 지도·관리', '불법게임기 보관창고 보수', '게임제공업소 등 지도단속',
    '불법사행성게임장 단속 압수물 관리', '게임 및 음악산업 진흥', '불법게임물 운반차량 임차비',
    '음악 및 게임산업 과징금 운용', '게임업 지도관리', '건전한 게임 문화 조성', '압수 불법 게임기 수거',
    '옥외광고물 문화개선', '옥외광고물 환경정비 사업', '옥외광고물 업무추진',
    '시사편찬', '강릉시 시사편찬', '부천시사편찬', '향토사료관 운영', '시사편찬위원회',
    '시사편찬 자료실 운영', '연천군지 편찬', '알기 쉬운 향토사 교육과정 운영',
    '인천시사편찬원 설립 타당성 검토 및 기본계획 수립 용역', '경주지역 향토사지 발간',
    '향토사 연구 및 자료 아카이빙', '향토사료 조사 지원',
    '불법 숙박 영업 행위 단속 추진', '유통관련업 지도단속', '유통관련 지도단속', '관광사업 지도점검',
    '농수산물 운임지원', '산림재해 일자리(산불)',
]
print('제외 대상 세부사업명:', len(SCOPE_EXCLUDE_PROJECTS), '개')

sub = reg[~reg['project_name'].isin(SCOPE_EXCLUDE_PROJECTS)].copy()
print('제외된 행 수:', (~reg.index.isin(sub.index)).sum(), '건 (scope_filter_proposal.html의 110건과 일치해야 함)')

agg = sub.groupby(['region', 'year'], as_index=False)['local_own_revenue'].sum()
agg['region_en'] = agg['region'].map(REGION_KR2EN)
agg = agg.rename(columns={'local_own_revenue': 'budget_excl'})

dfh2 = panel.merge(agg[['region_en', 'year', 'budget_excl']], left_on=['region', 'year'],
                    right_on=['region_en', 'year'], how='inner')

print('가설2 패널(연구범위 조정 반영) n =', len(dfh2))
dfh2.head()


## 3. 가설1 — 방문×지출 (TOP5)

세출예산 데이터를 쓰지 않으므로 연구범위 조정과 무관 — 1절에서 만든 `panel`(visit_w, spend_w)을 그대로 사용합니다.


### 3-1. ⑤ 지역+연도 고정효과 (Fixed Effects)

In [ ]:
dfh1 = panel.copy()
dfh1['visit_z'] = (dfh1['visit_w'] - dfh1['visit_w'].mean()) / dfh1['visit_w'].std()
dfh1['spend_z'] = (dfh1['spend_w'] - dfh1['spend_w'].mean()) / dfh1['spend_w'].std()

model_h1 = smf.ols('spend_z ~ visit_z + C(region) + C(year)', data=dfh1).fit(
    cov_type='cluster', cov_kwds={'groups': dfh1['region']})
beta_h1 = model_h1.params['visit_z']
p_h1 = model_h1.pvalues['visit_z']
ci_h1 = model_h1.conf_int().loc['visit_z']
print(f'H1 FE: beta={beta_h1:.4f}  p={p_h1:.4g}  95%CI=[{ci_h1[0]:.4f}, {ci_h1[1]:.4f}]')


### 3-2. ⑥ 연도별 변화량분석 (First-Difference) — Pearson + Spearman

In [ ]:
d1 = dfh1.sort_values(['region', 'year']).copy()
d1['dvisit'] = d1.groupby('region')['visit_z'].diff()
d1['dspend'] = d1.groupby('region')['spend_z'].diff()
d1 = d1.dropna(subset=['dvisit', 'dspend'])

r_h1, p_h1_diff = stats.pearsonr(d1['dvisit'], d1['dspend'])
rho_h1, ps_h1_diff = stats.spearmanr(d1['dvisit'], d1['dspend'])
print(f'H1 diff: n={len(d1)}  Pearson r={r_h1:.4f} (p={p_h1_diff:.4g})  |  Spearman rho={rho_h1:.4f} (p={ps_h1_diff:.4g})')


### 3-3. ⑧ 순열검정 (Permutation Test, 10,000회)

In [ ]:
rng = np.random.default_rng(RNG_SEED)
r_obs_h1, _ = stats.pearsonr(dfh1['visit_w'], dfh1['spend_w'])
perm_h1 = np.array([stats.pearsonr(dfh1['visit_w'].values, rng.permutation(dfh1['spend_w'].values))[0]
                     for _ in range(10000)])
p_perm_h1 = (np.sum(np.abs(perm_h1) >= abs(r_obs_h1)) + 1) / (10000 + 1)
print(f'H1 perm: obs_r={r_obs_h1:.4f}  p={p_perm_h1:.5f}')


### 3-4. ⑦ 부트스트랩 (지역 클러스터 부트스트랩, 5,000회)

In [ ]:
regions_h1 = dfh1['region'].unique()
boot_h1 = []
for _ in range(5000):
    samp = rng.choice(regions_h1, size=len(regions_h1), replace=True)
    rows_ = pd.concat([dfh1[dfh1['region'] == r] for r in samp], ignore_index=True)
    if rows_['visit_w'].std() > 0 and rows_['spend_w'].std() > 0:
        rb, _ = stats.pearsonr(rows_['visit_w'], rows_['spend_w'])
        boot_h1.append(rb)
ci_boot_h1 = np.percentile(boot_h1, [2.5, 97.5])
print(f'H1 bootstrap: obs_r={r_obs_h1:.4f}  95%CI=[{ci_boot_h1[0]:.4f}, {ci_boot_h1[1]:.4f}]')


### 3-5. ② 제주 제외 검증 (Leave-one-region-out)

In [ ]:
noje_h1 = dfh1[dfh1['region'] != 'jeju']
r_noje_h1, p_noje_h1 = stats.pearsonr(noje_h1['visit_w'], noje_h1['spend_w'])
print(f'H1 jeju: 전체 r={r_obs_h1:.4f}  ->  제주제외 r={r_noje_h1:.4f} (p={p_noje_h1:.4g})')


## 4. 가설2 — 방문×예산 (연구범위 조정 반영, TOP5)

### 4-1. ⑤ 지역+연도 고정효과 (Fixed Effects)

In [ ]:
dfh2['visit_z'] = (dfh2['visit_w'] - dfh2['visit_w'].mean()) / dfh2['visit_w'].std()
dfh2['budget_z'] = (dfh2['budget_excl'] - dfh2['budget_excl'].mean()) / dfh2['budget_excl'].std()

model_h2 = smf.ols('budget_z ~ visit_z + C(region) + C(year)', data=dfh2).fit(
    cov_type='cluster', cov_kwds={'groups': dfh2['region']})
beta_h2 = model_h2.params['visit_z']
p_h2 = model_h2.pvalues['visit_z']
ci_h2 = model_h2.conf_int().loc['visit_z']
print(f'H2 FE (연구범위 조정): beta={beta_h2:.4f}  p={p_h2:.4g}  95%CI=[{ci_h2[0]:.4f}, {ci_h2[1]:.4f}]')


### 4-2. ⑥ 연도별 변화량분석 — Pearson + Spearman

In [ ]:
d2 = dfh2.sort_values(['region', 'year']).copy()
d2['dvisit'] = d2.groupby('region')['visit_z'].diff()
d2['dbudget'] = d2.groupby('region')['budget_z'].diff()
d2 = d2.dropna(subset=['dvisit', 'dbudget'])

r_h2, p_h2_diff = stats.pearsonr(d2['dvisit'], d2['dbudget'])
rho_h2, ps_h2_diff = stats.spearmanr(d2['dvisit'], d2['dbudget'])
print(f'H2 diff (연구범위 조정): n={len(d2)}  Pearson r={r_h2:.4f} (p={p_h2_diff:.4g})  |  Spearman rho={rho_h2:.4f} (p={ps_h2_diff:.4g})')


### 4-3. ⑧ 순열검정 (10,000회)

In [ ]:
r_obs_h2, _ = stats.pearsonr(dfh2['visit_w'], dfh2['budget_excl'])
perm_h2 = np.array([stats.pearsonr(dfh2['visit_w'].values, rng.permutation(dfh2['budget_excl'].values))[0]
                     for _ in range(10000)])
p_perm_h2 = (np.sum(np.abs(perm_h2) >= abs(r_obs_h2)) + 1) / (10000 + 1)
print(f'H2 perm (연구범위 조정): obs_r={r_obs_h2:.4f}  p={p_perm_h2:.5f}')


### 4-4. ⑦ 부트스트랩 (지역 클러스터, 5,000회)

In [ ]:
regions_h2 = dfh2['region'].unique()
boot_h2 = []
for _ in range(5000):
    samp = rng.choice(regions_h2, size=len(regions_h2), replace=True)
    rows_ = pd.concat([dfh2[dfh2['region'] == r] for r in samp], ignore_index=True)
    if rows_['visit_w'].std() > 0 and rows_['budget_excl'].std() > 0:
        rb, _ = stats.pearsonr(rows_['visit_w'], rows_['budget_excl'])
        boot_h2.append(rb)
ci_boot_h2 = np.percentile(boot_h2, [2.5, 97.5])
print(f'H2 bootstrap (연구범위 조정): obs_r={r_obs_h2:.4f}  95%CI=[{ci_boot_h2[0]:.4f}, {ci_boot_h2[1]:.4f}]')


### 4-5. ② 제주 제외 검증

In [ ]:
noje_h2 = dfh2[dfh2['region'] != 'jeju']
r_noje_h2, p_noje_h2 = stats.pearsonr(noje_h2['visit_w'], noje_h2['budget_excl'])
print(f'H2 jeju (연구범위 조정): 전체 r={r_obs_h2:.4f}  ->  제주제외 r={r_noje_h2:.4f} (p={p_noje_h2:.4g})')


## 5. TOP5 결과 요약표

In [ ]:
summary = pd.DataFrame([
    ['⑤ 고정효과 β (p)', f'{beta_h1:.3f} ({p_h1:.3g})', f'{beta_h2:.3f} ({p_h2:.3g})'],
    ['⑥ 변화량분석 Pearson r (p)', f'{r_h1:.3f} ({p_h1_diff:.3g})', f'{r_h2:.3f} ({p_h2_diff:.3g})'],
    ['⑥ 변화량분석 Spearman ρ (p)', f'{rho_h1:.3f} ({ps_h1_diff:.3g})', f'{rho_h2:.3f} ({ps_h2_diff:.3g})'],
    ['⑧ 순열검정 p', f'{p_perm_h1:.4f}', f'{p_perm_h2:.4f}'],
    ['⑦ 부트스트랩 95%CI', f'[{ci_boot_h1[0]:.3f}, {ci_boot_h1[1]:.3f}]', f'[{ci_boot_h2[0]:.3f}, {ci_boot_h2[1]:.3f}]'],
    ['② 제주제외 r (전체→제외)', f'{r_obs_h1:.3f} → {r_noje_h1:.3f}', f'{r_obs_h2:.3f} → {r_noje_h2:.3f}'],
], columns=['방법', '가설1 (방문×지출)', '가설2 (방문×예산, 연구범위 조정)'])

summary


---
**해석 요약**
- 가설1: 065/연구범위 조정과 무관 — 이전과 동일하게 5개 방법 모두 지지 → 채택 (아래 참고치 그대로 유효)
- 가설2: 예산 정의가 바뀌었으므로(065 재포함 + 110건 제외) 아래 5개 방법의 숫자는 이 노트북을 실제로 돌려서 새로
  나오는 값을 확인해야 합니다. 예산 총액 변화가 지역별 최대 -0.13%로 작아서 결론(부분 채택) 자체가 뒤집힐 가능성은
  낮지만, "확인 없이 같다고 가정하지 않는다"는 원칙에 따라 숫자는 비워둡니다.

**검증 참고치 — 가설1만 (연구범위 조정과 무관하므로 그대로 유효)**

| 방법 | 가설1 |
|---|---|
| ⑤ FE β (p) | 0.754 (4.14e-14) |
| ⑥ Pearson r (p) | 0.783 (4.39e-08) |
| ⑥ Spearman ρ (p) | 0.776 (6.83e-08) |
| ⑧ 순열검정 p | 0.0001 |
| ⑦ 부트스트랩 95%CI | [0.368, 0.973] |
| ② 제주제외 r | 0.675→0.896 |

**가설2 참고치는 이번 개정으로 폐기됨** — 기존에 게재됐던 0.244/0.067/0.145 등의 수치는 065를 통째로 뺐던
구버전 정의 기준이라 지금 코드와 맞지 않습니다. 이 노트북을 실행해서 나온 새 숫자로 아티팩트를 갱신해야 합니다.


## 6. [부록] 방문당 자체재원 재정리 — 트랙1 전용 (트랙 분리 원칙 준수)

`정책_시사점_최종.pdf`의 '방문당 자체재원' 지표는 트랙1(연간·시/도 예산)과 트랙2(월별·1회차만 집계된 방문)를
결합해서 만들어졌습니다. 그런데 팀 방법론 문서(`국민여행조사_트랙별_분석방법론_정리.pdf`)와
`track_2_dataset` 컬럼정의서 모두 "두 트랙은 서로 재합산·교차검증하지 않는다"고 명시하고 있어, 원래 지표는
이 원칙과 어긋납니다(트랙2는 1~6차 중 1회차만 써서 전체보다 5~6% 낮게 집계되므로 방문량 자체도 과소평가됩니다).

아래는 같은 개념(지자체가 방문 1건당 자체 예산을 얼마나 쓰는가)을 **트랙1 데이터만으로** 다시 계산한 버전입니다.
방문은 위 1절에서 만든 `panel`(1~6차 전체 반영, 연간)의 `visit_w`를, 예산은 4절의 `dfh2`에서 쓴
`budget_excl`(관광 연구범위 조정 반영, 065 포함)을 그대로 사용합니다 — 새 파일이나 트랙2 데이터는 전혀 쓰지 않습니다.

이 표는 통계적 유의성 검정이 아니라 지역별 특징을 살펴보는 참고용 지표입니다. 상관관계·인과관계 해석에는
쓰지 않습니다.


In [ ]:
# 지역별 3개년 합산 (트랙1 전용: panel의 visit_w, dfh2의 budget_excl)
region_totals = dfh2.groupby('region', as_index=False).agg(
    visit_w_total=('visit_w', 'sum'),
    budget_excl_total=('budget_excl', 'sum'),
)
region_totals['방문당_자체재원'] = region_totals['budget_excl_total'] / region_totals['visit_w_total']

# 집행률 (연구범위 조정 반영, 3개년 합산) — processed_regional.csv에서 직접, 트랙2 데이터 사용 안 함
sub_all = reg[~reg['project_name'].isin(SCOPE_EXCLUDE_PROJECTS)]
exec_agg = sub_all.groupby('region', as_index=False).agg(
    budget_total=('budget_total', 'sum'), expenditure=('expenditure', 'sum'))
exec_agg['집행률'] = (exec_agg['expenditure'] / exec_agg['budget_total'] * 100).round(1)
exec_agg['region_en'] = exec_agg['region'].map(REGION_KR2EN)

# 지출 총액(spend_w, 트랙1)도 함께 붙여서 고방문/고지출 4분면 유형 재분류
spend_totals = panel.groupby('region', as_index=False)['spend_w'].sum()

result = (region_totals
          .merge(exec_agg[['region_en', '집행률']], left_on='region', right_on='region_en', how='left')
          .merge(spend_totals, on='region', how='left')
          .drop(columns='region_en'))

med_visit = result['visit_w_total'].median()
med_spend = result['spend_w'].median()
result['유형'] = np.where(result['visit_w_total'] >= med_visit, '고방문', '저방문') + '-' \
                 + np.where(result['spend_w'] >= med_spend, '고지출', '저지출')

result_display = result[['region', '유형', '방문당_자체재원', '집행률']].sort_values(
    '방문당_자체재원', ascending=False).reset_index(drop=True)
result_display
